# shap for AM-I

In [1]:
# ===== SVR MODEL SHAP ANALYSIS with Top-10 Large Font Visualization =====
import os
import joblib
import shap
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===== 设置大字体（采用第二个代码的规则） =====
FONT_SIZE = 50
TICK_SIZE = 50
TITLE_SIZE = 20
LEGEND_SIZE = 50

mpl.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': 600,
    'figure.dpi': 600
})

# ========== Configuration ==========
DATA_FOLDER = './1-train_test_split'
MODEL_FOLDER = './2-svr-models/AM-I-svr-model'
SHAP_FOLDER = './3-shap/AM-I-shap'
DRAWING_FOLDER = './3-shap/AM-I-shap/Top10'  # 专门保存Top-10图的目录
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

base_name = "AM-I-filtered_with_labels_k4"

# iPhone配色（保持第一个代码的风格）
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#FF3B30",
    "residual": "#34C759",
    "text": "#1C1C1E"
}

# ========== 1. Load Model, Scaler, and Training Data ==========
train_csv_path = os.path.join(DATA_FOLDER, f"{base_name}_train.csv")
test_csv_path = os.path.join(DATA_FOLDER, f"{base_name}_test.csv")
svr_model_path = os.path.join(MODEL_FOLDER, f"{base_name}_svr_model.joblib")
scaler_path = os.path.join(MODEL_FOLDER, f"{base_name}_scaler.joblib")

# Load SVR model and scaler
svr_model = joblib.load(svr_model_path)
scaler = joblib.load(scaler_path)

# Load training data
df_train = pd.read_csv(train_csv_path).dropna(subset=ALL_FEATURES + [TARGET_COL])
X_train = df_train[ALL_FEATURES].values
X_train[:, :len(FEATURE_COLS)] = scaler.transform(X_train[:, :len(FEATURE_COLS)])

# SVR predictions as surrogate model target
y_surrogate_train = svr_model.predict(X_train)

# ========== 2. Train LightGBM Surrogate Model ==========
lgb_train = lgb.Dataset(X_train, label=y_surrogate_train)
params = {
    'objective': 'regression',
    'metric': 'l2',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'num_leaves': 40,
    'learning_rate': 0.03,
    'n_estimators': 1500,
    'min_child_samples': 15,
    'reg_alpha': 0.05,
    'reg_lambda': 0.05,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'min_split_gain': 0.01,
    'min_child_weight': 0.001,
    'seed': SEED,
    'n_jobs': -1
}
gbm = lgb.train(params, lgb_train)

# Training set fidelity
y_hat_train = gbm.predict(X_train)
fidelity_r2_train = r2_score(y_surrogate_train, y_hat_train)
fidelity_rmse_train = np.sqrt(mean_squared_error(y_surrogate_train, y_hat_train))
fidelity_mae_train = mean_absolute_error(y_surrogate_train, y_hat_train)
print(f"Surrogate model fidelity on training set:")
print(f"  R²: {fidelity_r2_train:.4f}")
print(f"  RMSE: {fidelity_rmse_train:.4f}")
print(f"  MAE: {fidelity_mae_train:.4f}")

# ========== 3. Load Test Set and Compute Test Fidelity ==========
df_test = pd.read_csv(test_csv_path).dropna(subset=ALL_FEATURES + [TARGET_COL])
X_test = df_test[ALL_FEATURES].values
X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

y_surrogate_test = svr_model.predict(X_test)
y_hat_test = gbm.predict(X_test)
fidelity_r2_test = r2_score(y_surrogate_test, y_hat_test)
fidelity_rmse_test = np.sqrt(mean_squared_error(y_surrogate_test, y_hat_test))
fidelity_mae_test = mean_absolute_error(y_surrogate_test, y_hat_test)
print(f"\nSurrogate model fidelity on test set:")
print(f"  R²: {fidelity_r2_test:.4f}")
print(f"  RMSE: {fidelity_rmse_test:.4f}")
print(f"  MAE: {fidelity_mae_test:.4f}")

# ========== 4. Create Combined Performance Scatter Plot ==========
performance_folder = os.path.join(SHAP_FOLDER, "performance_plots")
os.makedirs(performance_folder, exist_ok=True)

# iPhone风格散点图（更新字号）
def create_iphone_style_plot(ax, x_data, y_data, title, metrics_text):
    ax.tick_params(axis='both', direction='out', length=6, width=1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.grid(False)
    
    ax.scatter(x_data, y_data, alpha=0.6, color=IPHONE_COLORS['scatter'], s=30, edgecolors='k', linewidth=0.5)
    ax.plot([x_data.min(), x_data.max()], [x_data.min(), x_data.max()], 
             linestyle='--', color=IPHONE_COLORS['line'], linewidth=2)
    
    ax.set_xlabel('SVR Predictions (s)', fontsize=FONT_SIZE, fontweight='bold', color=IPHONE_COLORS['text'])
    ax.set_ylabel('LightGBM Predictions (s)', fontsize=FONT_SIZE, fontweight='bold', color=IPHONE_COLORS['text'])
    
    # Title at bottom
    ax.text(0.5, -0.15, title, ha='center', va='center', transform=ax.transAxes, 
            fontsize=FONT_SIZE, color=IPHONE_COLORS['text'], fontweight='bold')
    
    # Metrics box
    ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, verticalalignment='top',
            fontsize=14, color=IPHONE_COLORS['text'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Combined scatter plot with iPhone style
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Training set
metrics_train = f'R² = {fidelity_r2_train:.3f}\nRMSE = {fidelity_rmse_train:.3f}\nMAE = {fidelity_mae_train:.3f}'
create_iphone_style_plot(axes[0], y_surrogate_train, y_hat_train, 
                         "Training Set", metrics_train)

# Test set with different color box
metrics_test = f'R² = {fidelity_r2_test:.3f}\nRMSE = {fidelity_rmse_test:.3f}\nMAE = {fidelity_mae_test:.3f}'
create_iphone_style_plot(axes[1], y_surrogate_test, y_hat_test, 
                         "Test Set", metrics_test)

plt.suptitle(f'Surrogate Model Fidelity: {base_name}', fontsize=TITLE_SIZE, fontweight='bold', 
             color=IPHONE_COLORS['text'])
plt.tight_layout()

# Save plots
scatter_png = os.path.join(performance_folder, f'{base_name}_surrogate_fidelity_scatter.png')
scatter_pdf = os.path.join(performance_folder, f'{base_name}_surrogate_fidelity_scatter.pdf')
plt.savefig(scatter_png, bbox_inches='tight', dpi=600)
plt.savefig(scatter_pdf, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance plots saved to: {performance_folder}")

# ========== 5. SHAP Analysis ==========
shap_analysis_folder = os.path.join(SHAP_FOLDER, "shap_analysis")
os.makedirs(shap_analysis_folder, exist_ok=True)
os.makedirs(DRAWING_FOLDER, exist_ok=True)  # 创建Drawing目录

explainer = shap.TreeExplainer(gbm, feature_perturbation="tree_path_dependent")
shap_values = explainer(X_train)

# Save SHAP values
np.save(os.path.join(shap_analysis_folder, f"{base_name}_shap_values.npy"), shap_values.values)
np.save(os.path.join(shap_analysis_folder, f"{base_name}_base_values.npy"), shap_values.base_values)
joblib.dump(shap_values, os.path.join(shap_analysis_folder, f"{base_name}_shap_explanation.joblib"))

# ========== 6. Calculate Mean Absolute SHAP Values, Select Top 20 ==========
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top20_idx = np.argsort(mean_abs_shap)[::-1][:20]

# Save top 20 features
df_top20 = pd.DataFrame({
    "Feature": [ALL_FEATURES[i] for i in top20_idx],
    "MeanAbsSHAP": [mean_abs_shap[i] for i in top20_idx]
})
df_top20.to_csv(os.path.join(shap_analysis_folder, f"{base_name}_top20_shap_features.csv"), index=False)

# ========== 7. 创建Top-10 SHAP summary plot（采用第二个代码的绘图规则）==========
# 计算Top-10特征的索引和名称
top10_idx = np.argsort(mean_abs_shap)[::-1][:10]
top10_names = [ALL_FEATURES[i] for i in top10_idx]

print("\n" + "="*50)
print("Top 10 Features by SHAP Importance:")
print("="*50)
for i, (idx, name) in enumerate(zip(top10_idx, top10_names), 1):
    print(f"{i:2d}. {name}: {mean_abs_shap[idx]:.6f}")
print("="*50 + "\n")

# 提取Top-10的SHAP值和数据
shap_values_top10 = shap_values.values[:, top10_idx]
data_top10 = shap_values.data[:, top10_idx]

# 创建Top-10 summary plot（严格按照第二个代码的风格）
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values_top10,
    data_top10,
    feature_names=top10_names,
    max_display=10,
    show=False,
    plot_size=None
)
plt.title(f'(a) AM-I', fontsize=TITLE_SIZE, fontweight='bold')  # 根据实际情况修改标题
plt.tight_layout()

# 保存到Drawing目录（第二个代码指定的目录）
drawing_png = os.path.join(DRAWING_FOLDER, f"{base_name}_shap_summary_top10.png")
drawing_pdf = os.path.join(DRAWING_FOLDER, f"{base_name}_shap_summary_top10.pdf")
plt.savefig(drawing_png, dpi=600, bbox_inches='tight')
plt.savefig(drawing_pdf, bbox_inches='tight')
plt.close()

# 同时也保存到shap_analysis文件夹
analysis_png = os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top10.png")
analysis_pdf = os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top10.pdf")
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values_top10,
    data_top10,
    feature_names=top10_names,
    max_display=10,
    show=False,
    plot_size=None
)
plt.title(f'Top 10 Feature Importance - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.tight_layout()
plt.savefig(analysis_png, dpi=600, bbox_inches='tight')
plt.savefig(analysis_pdf, bbox_inches='tight')
plt.close()

print(f"✅ Top-10 SHAP summary plot saved to Drawing directory:")
print(f"   PNG: {drawing_png}")
print(f"   PDF: {drawing_pdf}")

# ========== 8. 保存Top-10特征列表到CSV（采用第二个代码）==========
top10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Feature": top10_names,
    "MeanAbsSHAP": [mean_abs_shap[i] for i in top10_idx]
})
top10_csv_drawing = os.path.join(DRAWING_FOLDER, f"{base_name}_top10_features.csv")
top10_csv_analysis = os.path.join(shap_analysis_folder, f"{base_name}_top10_features.csv")
top10_df.to_csv(top10_csv_drawing, index=False)
top10_df.to_csv(top10_csv_analysis, index=False)
print(f"✅ Top-10 features list saved to: {top10_csv_drawing}")

# ========== 9. 可选：创建Top-10 dependence plots（大字体）==========
dependence_folder = os.path.join(DRAWING_FOLDER, "dependence_plots")
os.makedirs(dependence_folder, exist_ok=True)

for rank, (idx, name) in enumerate(zip(top10_idx, top10_names), 1):
    plt.figure(figsize=(10, 8))
    shap.dependence_plot(
        idx,
        shap_values.values,
        shap_values.data,
        feature_names=ALL_FEATURES,
        display_features=shap_values.data,
        show=False,
        alpha=0.7,
        dot_size=20
    )
    plt.title(f"Dependence: {name}", fontsize=TITLE_SIZE, fontweight='bold')
    plt.xlabel(name, fontsize=FONT_SIZE)
    plt.ylabel("SHAP value", fontsize=FONT_SIZE)
    plt.tight_layout()
    
    dep_png = os.path.join(dependence_folder, f"{base_name}_shap_dependence_top{rank}_{name}.png")
    dep_pdf = os.path.join(dependence_folder, f"{base_name}_shap_dependence_top{rank}_{name}.pdf")
    plt.savefig(dep_png, dpi=600, bbox_inches='tight')
    plt.savefig(dep_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ Dependence plot {rank}/10 → {dep_png}")

# ========== 10. 原代码的Top-20 summary plot（保持原样式）==========
X_train_top20 = X_train[:, top20_idx]
shap_values_top20 = shap_values.values[:, top20_idx]
feature_names_top20 = [ALL_FEATURES[i] for i in top20_idx]

plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values_top20,
    X_train_top20,
    feature_names=feature_names_top20,
    max_display=20,
    show=False,
    plot_size=None
)
plt.title(f'Top 20 Feature Importance - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top20.png"), 
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top20.pdf"), 
            bbox_inches='tight')
plt.close()

# ========== 11. 创建Top Features Bar Plot ==========
sorted_idx = np.argsort(mean_abs_shap)[::-1]
top_n = min(15, len(ALL_FEATURES))

plt.figure(figsize=(12, 10))
bars = plt.barh(range(top_n), mean_abs_shap[sorted_idx[:top_n]][::-1], 
                color=plt.cm.viridis(np.linspace(0.3, 0.9, top_n)))
plt.yticks(range(top_n), [ALL_FEATURES[i] for i in sorted_idx[:top_n]][::-1], fontsize=14)
plt.xlabel('Mean Absolute SHAP Value', fontsize=FONT_SIZE, fontweight='bold')
plt.title(f'Top {top_n} Most Important Features - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width * 1.01, bar.get_y() + bar.get_height()/2, 
             f'{width:.4f}', ha='left', va='center', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_feature_importance_bar.png"), 
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_feature_importance_bar.pdf"), 
            bbox_inches='tight')
plt.close()

# ========== 12. 创建force plot ==========
force_html = os.path.join(shap_analysis_folder, f"{base_name}_shap_force_sample0.html")
force_plot_obj = shap.force_plot(
    shap_values.base_values[0],
    shap_values.values[0],
    X_train[0, :],
    feature_names=ALL_FEATURES
)
shap.save_html(force_html, force_plot_obj)

# ========== 13. Save Summary Report ==========
summary_folder = os.path.join(SHAP_FOLDER, "summary_reports")
os.makedirs(summary_folder, exist_ok=True)

summary_df = pd.DataFrame({
    'Metric': ['Training R²', 'Training RMSE', 'Training MAE', 
               'Test R²', 'Test RMSE', 'Test MAE'],
    'Value': [fidelity_r2_train, fidelity_rmse_train, fidelity_mae_train,
              fidelity_r2_test, fidelity_rmse_test, fidelity_mae_test]
})
summary_csv = os.path.join(summary_folder, f"{base_name}_surrogate_model_summary.csv")
summary_df.to_csv(summary_csv, index=False)

# Print detailed fidelity metrics to text file
fidelity_report = os.path.join(summary_folder, f"{base_name}_fidelity_report.txt")
with open(fidelity_report, 'w') as f:
    f.write(f"Surrogate Model Fidelity Report: {base_name}\n")
    f.write("=" * 50 + "\n\n")
    f.write("Training Set Performance:\n")
    f.write(f"  R²: {fidelity_r2_train:.6f}\n")
    f.write(f"  RMSE: {fidelity_rmse_train:.6f}\n")
    f.write(f"  MAE: {fidelity_mae_train:.6f}\n\n")
    f.write("Test Set Performance:\n")
    f.write(f"  R²: {fidelity_r2_test:.6f}\n")
    f.write(f"  RMSE: {fidelity_rmse_test:.6f}\n")
    f.write(f"  MAE: {fidelity_mae_test:.6f}\n\n")
    f.write("Top 10 Features by Mean Absolute SHAP:\n")
    for i, idx in enumerate(sorted_idx[:10]):
        f.write(f"  {i+1}. {ALL_FEATURES[idx]}: {mean_abs_shap[idx]:.6f}\n")

# ========== 14. Save Model Metadata ==========
metadata_folder = os.path.join(SHAP_FOLDER, "model_metadata")
os.makedirs(metadata_folder, exist_ok=True)

# Save LightGBM model
lgb_model_path = os.path.join(metadata_folder, f"{base_name}_lightgbm_model.txt")
gbm.save_model(lgb_model_path)

# Save training parameters
params_df = pd.DataFrame([params])
params_df.to_csv(os.path.join(metadata_folder, f"{base_name}_model_params.csv"), index=False)

# ========== 15. Print Summary ==========
print(f"\n✅ SHAP analysis completed. Results saved to: {shap_analysis_folder}")
print(f"✅ Performance plots saved to: {performance_folder}")
print(f"✅ Summary reports saved to: {summary_folder}")
print(f"✅ Model metadata saved to: {metadata_folder}")
print(f"✅ Top-10 plots saved to Drawing directory: {DRAWING_FOLDER}")
print(f"✅ Dependence plots saved to: {dependence_folder}")
print(f"✅ All plots saved in PNG and PDF formats with publication quality.")

# ========== 16. Directory Structure Summary ==========
print(f"\n📁 Complete Directory Structure:")
print(f"├── {DATA_FOLDER}/")
print(f"│   ├── {base_name}_train.csv")
print(f"│   └── {base_name}_test.csv")
print(f"├── {MODEL_FOLDER}/")
print(f"│   ├── {base_name}_svr_model.joblib")
print(f"│   └── {base_name}_scaler.joblib")
print(f"├── {SHAP_FOLDER}/")
print(f"│   ├── performance_plots/")
print(f"│   ├── shap_analysis/")
print(f"│   │   ├── {base_name}_shap_values.npy")
print(f"│   │   ├── {base_name}_shap_explanation.joblib")
print(f"│   │   ├── {base_name}_shap_summary_top10.png/pdf")
print(f"│   │   ├── {base_name}_shap_summary_top20.png/pdf")
print(f"│   │   └── {base_name}_top10_features.csv")
print(f"│   ├── summary_reports/")
print(f"│   └── model_metadata/")
print(f"└── {DRAWING_FOLDER}/")
print(f"    ├── {base_name}_shap_summary_top10.png/pdf")
print(f"    ├── {base_name}_top10_features.csv")
print(f"    └── dependence_plots/")
print(f"        └── {base_name}_shap_dependence_top*_*.png/pdf")

# ========== 17. Print Quick Stats ==========
print(f"\n📊 Quick Statistics:")
print(f"   Training set size: {len(X_train)} samples")
print(f"   Test set size: {len(X_test)} samples")
print(f"   Total features: {len(ALL_FEATURES)}")
print(f"   Training fidelity R²: {fidelity_r2_train:.4f}")
print(f"   Test fidelity R²: {fidelity_r2_test:.4f}")

# ========== 18. Save the complete code for reproducibility ==========
code_save_folder = os.path.join(SHAP_FOLDER, "code_backup")
os.makedirs(code_save_folder, exist_ok=True)

print(f"\n💾 Code backup folder created: {code_save_folder}")
print("   (Manually save this script for reproducibility)")

# ========== 19. Final Completion Message ==========
print("\n" + "="*60)
print("🎉 恭喜！SHAP分析及Top-10可视化图生成完成！")
print("="*60)
print(f"📊 Top-10 SHAP summary plot已保存至: {DRAWING_FOLDER}")
print(f"📊 特征重要性列表已保存至: {top10_csv_drawing}")
print(f"📊 字体大小设置: 轴标签={FONT_SIZE}, 刻度={TICK_SIZE}, 标题={TITLE_SIZE}")
print(f"📁 所有文件均已保存为PNG和PDF格式")
print("="*60)

/home/xuxianyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Surrogate model fidelity on training set:
  R²: 0.9888
  RMSE: 1.3961
  MAE: 1.0612

Surrogate model fidelity on test set:
  R²: 0.9555
  RMSE: 2.6433
  MAE: 1.9853

✅ Performance plots saved to: ./3-shap/AM-I-shap/performance_plots

Top 10 Features by SHAP Importance:
 1. logP: 5.410100
 2. H_bond_donors: 1.981981
 3. fp_842: 1.568083
 4. MolWt: 1.000310
 5. col350: 0.851176
 6. fp_579: 0.838063
 7. fp_707: 0.773400
 8. col808: 0.714676
 9. col316: 0.678439
10. fp_145: 0.549696

✅ Top-10 SHAP summary plot saved to Drawing directory:
   PNG: ./3-shap/AM-I-shap/Top10/AM-I-filtered_with_labels_k4_shap_summary_top10.png
   PDF: ./3-shap/AM-I-shap/Top10/AM-I-filtered_with_labels_k4_shap_summary_top10.pdf
✅ Top-10 features list saved to: ./3-shap/AM-I-shap/Top10/AM-I-filtered_with_labels_k4_top10_features.csv
✅ Dependence plot 1/10 → ./3-shap/AM-I-shap/Top10/dependence_plots/AM-I-filtered_with_labels_k4_shap_dependence_top1_logP.png
✅ Dependence plot 2/10 → ./3-shap/AM-I-shap/Top10/dependen

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

...

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

# shap for AM-II

In [2]:
# SVR MODEL SHAP ANALYSIS (Surrogate LightGBM) - Overfit-reduced version for AM-II-shap-3
# Key changes:
# 1) Add validation split + early stopping (most effective)
# 2) Use LGBMRegressor wrapper for clean training/inference
# 3) Slightly stronger regularization & constraints to reduce memorization
# 4) Keep publication-quality plots & SHAP outputs
# 5) Modified plotting style with large fonts and specific output directory

import os
import joblib
import shap
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# ===== 超大字体设置 (遵循第二个代码的绘图规则) =====
FONT_SIZE = 50
TICK_SIZE = 50
TITLE_SIZE = 20  # 标题字体保持较小
LEGEND_SIZE = 50

mpl.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': 600,
    'figure.dpi': 600  # 改为600，遵循第二个代码
})

# iPhone-like colors (保留第一个代码的颜色方案)
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#FF3B30",
    "residual": "#34C759",
    "text": "#1C1C1E"
}

# ========== 配置参数 ==========
DATA_FOLDER = './1-train_test_split'
MODEL_FOLDER = './2-svr-models/AM-II-svr-model'
SHAP_FOLDER = './3-shap/AM-II-shap'  # SHAP输出主文件夹
DRAWING_FOLDER = './3-shap/AM-II-shap/Top10'    # 专门用于绘图的输出目录（遵循第二个代码）
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

base_name = "AM-II-filtered_with_labels_k4"

# ========== 1. 加载模型、Scaler和训练数据 ==========
train_csv_path = os.path.join(DATA_FOLDER, f"{base_name}_train.csv")
test_csv_path = os.path.join(DATA_FOLDER, f"{base_name}_test.csv")
svr_model_path = os.path.join(MODEL_FOLDER, f"{base_name}_svr_model.joblib")
scaler_path = os.path.join(MODEL_FOLDER, f"{base_name}_scaler.joblib")

# 加载SVR模型和scaler
svr_model = joblib.load(svr_model_path)
scaler = joblib.load(scaler_path)

# 加载训练数据
df_train = pd.read_csv(train_csv_path).dropna(subset=ALL_FEATURES + [TARGET_COL])
X_train = df_train[ALL_FEATURES].values
X_train[:, :len(FEATURE_COLS)] = scaler.transform(X_train[:, :len(FEATURE_COLS)])

# 代理目标 = SVR预测值
y_surrogate_train = svr_model.predict(X_train)

# 分割验证集用于早停
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_surrogate_train, test_size=0.15, random_state=SEED
)

# ========== 2. 训练LightGBM代理模型（过拟合优化） ==========
lgbm = lgb.LGBMRegressor(
    objective='regression',
    boosting_type='gbdt',
    learning_rate=0.04,
    n_estimators=1500,
    num_leaves=25,
    min_child_samples=15,
    min_split_gain=0.01,
    min_child_weight=1e-3,
    subsample=0.75,
    subsample_freq=1,
    colsample_bytree=0.75,
    reg_alpha=0.05,
    reg_lambda=0.05,
    random_state=SEED,
    n_jobs=-1
)

# 使用早停训练
lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='l2',
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

best_iter = getattr(lgbm, "best_iteration_", None)
print(f"\n最佳迭代次数（早停）: {best_iter}")

# 训练集保真度评估
y_hat_train = lgbm.predict(X_train, num_iteration=best_iter)
fidelity_r2_train = r2_score(y_surrogate_train, y_hat_train)
fidelity_rmse_train = np.sqrt(mean_squared_error(y_surrogate_train, y_hat_train))
fidelity_mae_train = mean_absolute_error(y_surrogate_train, y_hat_train)
print(f"\n代理模型在训练集上的保真度:")
print(f"  R²:   {fidelity_r2_train:.4f}")
print(f"  RMSE: {fidelity_rmse_train:.4f}")
print(f"  MAE:  {fidelity_mae_train:.4f}")

# ========== 3. 加载测试集并评估测试保真度 ==========
df_test = pd.read_csv(test_csv_path).dropna(subset=ALL_FEATURES + [TARGET_COL])
X_test = df_test[ALL_FEATURES].values
X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

y_surrogate_test = svr_model.predict(X_test)
y_hat_test = lgbm.predict(X_test, num_iteration=best_iter)

fidelity_r2_test = r2_score(y_surrogate_test, y_hat_test)
fidelity_rmse_test = np.sqrt(mean_squared_error(y_surrogate_test, y_hat_test))
fidelity_mae_test = mean_absolute_error(y_surrogate_test, y_hat_test)
print(f"\n代理模型在测试集上的保真度:")
print(f"  R²:   {fidelity_r2_test:.4f}")
print(f"  RMSE: {fidelity_rmse_test:.4f}")
print(f"  MAE:  {fidelity_mae_test:.4f}")

# ========== 4. 创建保真度散点图 ==========
performance_folder = os.path.join(SHAP_FOLDER, "performance_plots")
os.makedirs(performance_folder, exist_ok=True)

def create_iphone_style_plot(ax, x_data, y_data, title, metrics_text):
    """创建iPhone风格的散点图（字体使用当前设置）"""
    ax.tick_params(axis='both', direction='out', length=6, width=1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.grid(False)

    ax.scatter(
        x_data, y_data,
        alpha=0.6, color=IPHONE_COLORS['scatter'],
        s=30, edgecolors='k', linewidth=0.5
    )
    ax.plot(
        [x_data.min(), x_data.max()],
        [x_data.min(), x_data.max()],
        linestyle='--', color=IPHONE_COLORS['line'], linewidth=2
    )

    ax.set_xlabel('SVR Predictions (s)', fontweight='bold', color=IPHONE_COLORS['text'])
    ax.set_ylabel('LightGBM Predictions (s)', fontweight='bold', color=IPHONE_COLORS['text'])

    ax.text(
        0.5, -0.15, title,
        ha='center', va='center', transform=ax.transAxes,
        fontsize=FONT_SIZE, color=IPHONE_COLORS['text'], fontweight='bold'
    )

    ax.text(
        0.05, 0.95, metrics_text,
        transform=ax.transAxes, verticalalignment='top',
        fontsize=40, color=IPHONE_COLORS['text'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    )

fig, axes = plt.subplots(1, 2, figsize=(24, 10), sharey=True)

metrics_train = f'R² = {fidelity_r2_train:.3f}\nRMSE = {fidelity_rmse_train:.3f}\nMAE = {fidelity_mae_train:.3f}'
create_iphone_style_plot(axes[0], y_surrogate_train, y_hat_train, "Training Set", metrics_train)

metrics_test = f'R² = {fidelity_r2_test:.3f}\nRMSE = {fidelity_rmse_test:.3f}\nMAE = {fidelity_mae_test:.3f}'
create_iphone_style_plot(axes[1], y_surrogate_test, y_hat_test, "Test Set", metrics_test)

plt.suptitle(f'Surrogate Model Fidelity (Overfit-reduced): {base_name}', fontsize=TITLE_SIZE, fontweight='bold',
             color=IPHONE_COLORS['text'])
plt.tight_layout()

scatter_png = os.path.join(performance_folder, f'{base_name}_surrogate_fidelity_scatter.png')
scatter_pdf = os.path.join(performance_folder, f'{base_name}_surrogate_fidelity_scatter.pdf')
plt.savefig(scatter_png, bbox_inches='tight', dpi=600)
plt.savefig(scatter_pdf, bbox_inches='tight')
plt.close()

print(f"\n✅ 保真度散点图已保存至: {performance_folder}")

# ========== 5. SHAP分析 ==========
shap_analysis_folder = os.path.join(SHAP_FOLDER, "shap_analysis")
os.makedirs(shap_analysis_folder, exist_ok=True)

# 使用底层Booster进行TreeExplainer
booster = lgbm.booster_

explainer = shap.TreeExplainer(booster, feature_perturbation="tree_path_dependent")
shap_values = explainer(X_train)

# 保存SHAP值
np.save(os.path.join(shap_analysis_folder, f"{base_name}_shap_values.npy"), shap_values.values)
np.save(os.path.join(shap_analysis_folder, f"{base_name}_base_values.npy"), shap_values.base_values)
joblib.dump(shap_values, os.path.join(shap_analysis_folder, f"{base_name}_shap_explanation.joblib"))

# ========== 6. 计算平均|SHAP|并获取Top-20特征 ==========
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top20_idx = np.argsort(mean_abs_shap)[::-1][:20]

df_top20 = pd.DataFrame({
    "Feature": [ALL_FEATURES[i] for i in top20_idx],
    "MeanAbsSHAP": [mean_abs_shap[i] for i in top20_idx]
})
df_top20.to_csv(os.path.join(shap_analysis_folder, f"{base_name}_top20_shap_features.csv"), index=False)

# ========== 7. 创建Top-10 summary plot - 遵循第二个代码的绘图规则 ==========
# 创建Drawing目录（遵循第二个代码）
os.makedirs(DRAWING_FOLDER, exist_ok=True)

# 获取Top-10特征的索引和名称
top10_idx = np.argsort(mean_abs_shap)[::-1][:10]
top10_names = [ALL_FEATURES[i] for i in top10_idx]

print("\nTop 10 Features by SHAP Importance:")
for i, (idx, name) in enumerate(zip(top10_idx, top10_names), 1):
    print(f"{i:2d}. {name}: {mean_abs_shap[idx]:.6f}")

# 提取Top-10的SHAP值和数据
shap_values_top10 = shap_values.values[:, top10_idx]
data_top10 = shap_values.data[:, top10_idx]

# 创建Top-10 summary plot - 完全遵循第二个代码的样式
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values_top10,
    data_top10,
    feature_names=top10_names,
    max_display=10,
    show=False,
    plot_size=None
)
# 标题格式使用'(b) AM-II'，遵循第二个代码
plt.title(f'(b) AM-II', fontsize=TITLE_SIZE, fontweight='bold')
plt.tight_layout()

# 保存到Drawing目录（遵循第二个代码）
drawing_png = os.path.join(DRAWING_FOLDER, f"{base_name}_shap_summary_top10.png")
drawing_pdf = os.path.join(DRAWING_FOLDER, f"{base_name}_shap_summary_top10.pdf")

plt.savefig(drawing_png, dpi=600, bbox_inches='tight')
plt.savefig(drawing_pdf, bbox_inches='tight')
plt.close()

print(f"\n✅ Top-10 SHAP summary plot saved to Drawing directory:")
print(f"   PNG: {drawing_png}")
print(f"   PDF: {drawing_pdf}")

# 同时保存一份到shap_analysis文件夹（备份）
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values_top10,
    data_top10,
    feature_names=top10_names,
    max_display=10,
    show=False,
    plot_size=None
)
plt.title(f'(c) AM-II', fontsize=TITLE_SIZE, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top10.png"), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top10.pdf"), bbox_inches='tight')
plt.close()

# ========== 8. 保存Top-10特征列表到CSV（遵循第二个代码） ==========
top10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Feature": top10_names,
    "MeanAbsSHAP": [mean_abs_shap[i] for i in top10_idx]
})
top10_csv = os.path.join(DRAWING_FOLDER, f"{base_name}_top10_features.csv")
top10_df.to_csv(top10_csv, index=False)
print(f"✅ Top-10 features list saved to: {top10_csv}")

# 同时保存一份到shap_analysis文件夹
top10_csv_backup = os.path.join(shap_analysis_folder, f"{base_name}_top10_features.csv")
top10_df.to_csv(top10_csv_backup, index=False)

# ========== 9. 依赖图（保持原有样式，但使用大字体） ==========
large_font_folder = os.path.join(shap_analysis_folder, "large_font_plots")
os.makedirs(large_font_folder, exist_ok=True)

# 为Top-10特征创建依赖图
for rank, (idx, name) in enumerate(zip(top10_idx, top10_names), 1):
    plt.figure(figsize=(10, 8))
    shap.dependence_plot(
        idx,
        shap_values.values,
        shap_values.data,
        feature_names=ALL_FEATURES,
        display_features=shap_values.data,
        show=False,
        alpha=0.7,
        dot_size=20
    )
    plt.title(f"Dependence: {name}", fontsize=TITLE_SIZE, fontweight='bold')
    plt.xlabel(name, fontsize=FONT_SIZE)
    plt.ylabel("SHAP value", fontsize=FONT_SIZE)
    plt.tight_layout()

    out_dep_png = os.path.join(large_font_folder, f"{base_name}_shap_dependence_top{rank}_{name}_large_font.png")
    out_dep_pdf = os.path.join(large_font_folder, f"{base_name}_shap_dependence_top{rank}_{name}_large_font.pdf")
    plt.savefig(out_dep_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_dep_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ Dependence plot {rank}/10 → {out_dep_png}")

# ========== 10. Top-20 summary plot ==========
X_train_top20 = X_train[:, top20_idx]
shap_values_top20 = shap_values.values[:, top20_idx]
feature_names_top20 = [ALL_FEATURES[i] for i in top20_idx]

plt.figure(figsize=(14, 12))
shap.summary_plot(
    shap_values_top20,
    X_train_top20,
    feature_names=feature_names_top20,
    max_display=20,
    show=False,
    plot_size=None
)
plt.title(f'Top 20 Feature Importance - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top20.png"),
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_shap_summary_top20.pdf"),
            bbox_inches='tight')
plt.close()

# ========== 11. 特征重要性条形图（Top15） ==========
sorted_idx = np.argsort(mean_abs_shap)[::-1]
top_n = min(15, len(ALL_FEATURES))

plt.figure(figsize=(14, 12))
bars = plt.barh(range(top_n), mean_abs_shap[sorted_idx[:top_n]][::-1],
                color=plt.cm.viridis(np.linspace(0.3, 0.9, top_n)))
plt.yticks(range(top_n), [ALL_FEATURES[i] for i in sorted_idx[:top_n]][::-1], fontsize=40)
plt.xlabel('Mean Absolute SHAP Value', fontsize=FONT_SIZE, fontweight='bold')
plt.title(f'Top {top_n} Most Important Features - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for bar in bars:
    width = bar.get_width()
    plt.text(width * 1.01, bar.get_y() + bar.get_height()/2,
             f'{width:.4f}', ha='left', va='center', fontsize=30)

plt.tight_layout()
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_feature_importance_bar.png"),
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(shap_analysis_folder, f"{base_name}_feature_importance_bar.pdf"),
            bbox_inches='tight')
plt.close()

# ========== 12. 大字体条形图（Top20） ==========
plt.figure(figsize=(16, 14))
top_bar_n = min(20, len(ALL_FEATURES))
bars = plt.barh(range(top_bar_n), mean_abs_shap[sorted_idx[:top_bar_n]][::-1],
                color=plt.cm.plasma(np.linspace(0.2, 0.8, top_bar_n)))
plt.yticks(range(top_bar_n), [ALL_FEATURES[i] for i in sorted_idx[:top_bar_n]][::-1], fontsize=40)
plt.xlabel('Mean Absolute SHAP Value', fontsize=FONT_SIZE, fontweight='bold')
plt.ylabel('Features', fontsize=FONT_SIZE, fontweight='bold')
plt.title(f'Top {top_bar_n} Feature Importance - {base_name}', fontsize=TITLE_SIZE, fontweight='bold')
plt.grid(axis='x', alpha=0.3, linestyle='--')

for bar in bars:
    width = bar.get_width()
    plt.text(width * 1.01, bar.get_y() + bar.get_height()/2,
             f'{width:.4f}', ha='left', va='center', fontsize=30, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(large_font_folder, f"{base_name}_feature_importance_bar_large_font.png"),
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(large_font_folder, f"{base_name}_feature_importance_bar_large_font.pdf"),
            bbox_inches='tight')
plt.close()

# ========== 13. force plot样例 ==========
force_html = os.path.join(shap_analysis_folder, f"{base_name}_shap_force_sample0.html")
force_plot_obj = shap.force_plot(
    shap_values.base_values[0],
    shap_values.values[0],
    X_train[0, :],
    feature_names=ALL_FEATURES
)
shap.save_html(force_html, force_plot_obj)

# ========== 14. 保存摘要报告 ==========
summary_folder = os.path.join(SHAP_FOLDER, "summary_reports")
os.makedirs(summary_folder, exist_ok=True)

summary_df = pd.DataFrame({
    'Metric': ['Training R²', 'Training RMSE', 'Training MAE',
               'Test R²', 'Test RMSE', 'Test MAE', 'Best iteration'],
    'Value': [fidelity_r2_train, fidelity_rmse_train, fidelity_mae_train,
              fidelity_r2_test, fidelity_rmse_test, fidelity_mae_test, best_iter]
})
summary_csv = os.path.join(summary_folder, f"{base_name}_surrogate_model_summary.csv")
summary_df.to_csv(summary_csv, index=False)

fidelity_report = os.path.join(summary_folder, f"{base_name}_fidelity_report.txt")
with open(fidelity_report, 'w') as f:
    f.write(f"Surrogate Model Fidelity Report (Overfit-reduced): {base_name}\n")
    f.write("=" * 60 + "\n\n")
    f.write("Early stopping:\n")
    f.write(f"  Best iteration: {best_iter}\n\n")
    f.write("Training Set Performance:\n")
    f.write(f"  R²:   {fidelity_r2_train:.6f}\n")
    f.write(f"  RMSE: {fidelity_rmse_train:.6f}\n")
    f.write(f"  MAE:  {fidelity_mae_train:.6f}\n\n")
    f.write("Test Set Performance:\n")
    f.write(f"  R²:   {fidelity_r2_test:.6f}\n")
    f.write(f"  RMSE: {fidelity_rmse_test:.6f}\n")
    f.write(f"  MAE:  {fidelity_mae_test:.6f}\n\n")
    f.write("Top 10 Features by Mean Absolute SHAP:\n")
    for i, idx in enumerate(sorted_idx[:10]):
        f.write(f"  {i+1}. {ALL_FEATURES[idx]}: {mean_abs_shap[idx]:.6f}\n")

# ========== 15. 保存模型元数据 ==========
metadata_folder = os.path.join(SHAP_FOLDER, "model_metadata")
os.makedirs(metadata_folder, exist_ok=True)

# 保存booster模型
lgb_model_path = os.path.join(metadata_folder, f"{base_name}_lightgbm_model.txt")
booster.save_model(lgb_model_path)

params_df = pd.DataFrame([lgbm.get_params()])
params_df.to_csv(os.path.join(metadata_folder, f"{base_name}_model_params.csv"), index=False)

# 保存拟合的代理模型
joblib.dump(lgbm, os.path.join(metadata_folder, f"{base_name}_lgbm_regressor.joblib"))

# ========== 16. 打印摘要 ==========
print(f"\n✅ SHAP分析完成。结果保存至: {shap_analysis_folder}")
print(f"✅ 保真度图保存至: {performance_folder}")
print(f"✅ 摘要报告保存至: {summary_folder}")
print(f"✅ 模型元数据保存至: {metadata_folder}")
print(f"✅ 大字体图保存至: {large_font_folder}")
print(f"✅ Top-10 SHAP summary plot保存至: {DRAWING_FOLDER}")

# ========== 17. 目录结构摘要 ==========
print(f"\n📁 完整目录结构:")
print(f"├── {DATA_FOLDER}/")
print(f"│   ├── {base_name}_train.csv")
print(f"│   └── {base_name}_test.csv")
print(f"├── {MODEL_FOLDER}/")
print(f"│   ├── {base_name}_svr_model.joblib")
print(f"│   └── {base_name}_scaler.joblib")
print(f"├── {SHAP_FOLDER}/")
print(f"│   ├── performance_plots/")
print(f"│   ├── shap_analysis/")
print(f"│   ├── summary_reports/")
print(f"│   └── model_metadata/")
print(f"└── {DRAWING_FOLDER}/")
print(f"    ├── {base_name}_shap_summary_top10.png")
print(f"    ├── {base_name}_shap_summary_top10.pdf")
print(f"    └── {base_name}_top10_features.csv")

# ========== 18. 快速统计 ==========
print(f"\n📊 快速统计:")
print(f"   训练集大小: {len(X_train)} 样本")
print(f"   测试集大小: {len(X_test)} 样本")
print(f"   总特征数: {len(ALL_FEATURES)}")
print(f"   训练保真度 R²: {fidelity_r2_train:.4f}")
print(f"   测试保真度 R²: {fidelity_r2_test:.4f}")
print(f"   最佳迭代次数: {best_iter}")

# ========== 19. 代码备份文件夹 ==========
code_save_folder = os.path.join(SHAP_FOLDER, "code_backup")
os.makedirs(code_save_folder, exist_ok=True)
print(f"\n💾 代码备份文件夹已创建: {code_save_folder}")
print("   (请手动保存此脚本以保证可重复性)")
print("\n🎉 过拟合优化的代理模型 + SHAP图生成完成！")
print(f"🎯 关键输出: Top-10 SHAP summary plot已保存至 {DRAWING_FOLDER} (遵循第二个代码的绘图规则)")

Training until validation scores don't improve for 200 rounds
[100]	valid_0's l2: 17.9994
[200]	valid_0's l2: 14.3119
[300]	valid_0's l2: 13.2595
[400]	valid_0's l2: 12.6178
[500]	valid_0's l2: 12.3797
[600]	valid_0's l2: 12.2556
[700]	valid_0's l2: 12.0199
[800]	valid_0's l2: 11.8946
[900]	valid_0's l2: 11.8494
[1000]	valid_0's l2: 11.7885
[1100]	valid_0's l2: 11.7272
[1200]	valid_0's l2: 11.6629
[1300]	valid_0's l2: 11.6388
[1400]	valid_0's l2: 11.6271
[1500]	valid_0's l2: 11.5901
Did not meet early stopping. Best iteration is:
[1493]	valid_0's l2: 11.5892

最佳迭代次数（早停）: 1493

代理模型在训练集上的保真度:
  R²:   0.9798
  RMSE: 1.3453
  MAE:  0.5531


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



代理模型在测试集上的保真度:
  R²:   0.9422
  RMSE: 2.0648
  MAE:  1.5608

✅ 保真度散点图已保存至: ./3-shap/AM-II-shap/performance_plots

Top 10 Features by SHAP Importance:
 1. logP: 3.511303
 2. fp_842: 1.615719
 3. MolWt: 0.826814
 4. H_bond_donors: 0.519676
 5. fp_579: 0.468032
 6. fp_707: 0.466671
 7. col753: 0.411595
 8. fp_902: 0.408379
 9. col316: 0.357485
10. TPSA: 0.353158

✅ Top-10 SHAP summary plot saved to Drawing directory:
   PNG: ./3-shap/AM-II-shap/Top10/AM-II-filtered_with_labels_k4_shap_summary_top10.png
   PDF: ./3-shap/AM-II-shap/Top10/AM-II-filtered_with_labels_k4_shap_summary_top10.pdf
✅ Top-10 features list saved to: ./3-shap/AM-II-shap/Top10/AM-II-filtered_with_labels_k4_top10_features.csv
✅ Dependence plot 1/10 → ./3-shap/AM-II-shap/shap_analysis/large_font_plots/AM-II-filtered_with_labels_k4_shap_dependence_top1_logP_large_font.png
✅ Dependence plot 2/10 → ./3-shap/AM-II-shap/shap_analysis/large_font_plots/AM-II-filtered_with_labels_k4_shap_dependence_top2_fp_842_large_font.png
✅ D

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

...

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

<Figure size 6000x4800 with 0 Axes>

# shap analysis for AM-III, AM-IV, AM-V, AM-VI

In [3]:
import os
import joblib
import shap
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import r2_score

# ===== 设置大字体（参考第二段代码）=====
FONT_SIZE = 50
TICK_SIZE = 50
TITLE_SIZE = 20
LEGEND_SIZE = 50

mpl.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': 600,
    'figure.dpi': 600
})

# ========== 配置 ==========
DATA_FOLDER = "./processed_results"                 # 数据目录（和预训练模型一致）
MODEL_FOLDER = "./2-svr-model-other4"   # 模型目录
OUTPUT_FOLDER = "./3-shap-other4"               # SHAP输出目录
DRAWING_FOLDER = "./3-shap-other4/Top10"               # 绘图输出目录（参考第二段代码）
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

FILE_PREFIXES = [
    "AM-III-filtered",
    "AM-IV-filtered",
    "AM-V-filtered", 
    "AM-VI-filtered"
]

# 创建SHAP输出目录和绘图目录
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(DRAWING_FOLDER, exist_ok=True)

fidelity_records = []  # 存储所有 fidelity 结果

# ========== 分析逻辑 ==========
for prefix in FILE_PREFIXES:
    # 提取数据集名称（去掉-filtered后缀）
    dataset_name = prefix.replace("-filtered", "")
    print(f"\n===== 开始 SHAP 分析: {dataset_name} ({prefix}) =====")

    svr_model_path = os.path.join(MODEL_FOLDER, f"{prefix}_final_svr_model.joblib")
    scaler_path = os.path.join(MODEL_FOLDER, f"{prefix}_final_scaler.joblib")

    if not os.path.exists(svr_model_path):
        print(f"⚠️ 跳过 {dataset_name}，找不到模型文件: {svr_model_path}")
        continue

    svr_model = joblib.load(svr_model_path)
    scaler = joblib.load(scaler_path)

    # 加载数据
    csv_path = os.path.join(DATA_FOLDER, f"{prefix}.csv")
    if not os.path.exists(csv_path):
        print(f"⚠️ 跳过 {dataset_name}，找不到 CSV 数据: {csv_path}")
        continue

    df = pd.read_csv(csv_path).dropna(subset=ALL_FEATURES + [TARGET_COL])
    if df.shape[0] == 0:
        print(f"⚠️ 跳过 {dataset_name}，数据为空")
        continue

    X = df[ALL_FEATURES].values
    X[:, :len(FEATURE_COLS)] = scaler.transform(X[:, :len(FEATURE_COLS)])
    y_svr = svr_model.predict(X)

    # surrogate LightGBM
    lgb_train = lgb.Dataset(X, label=y_svr)
    params = {
        'objective': 'regression',
        'metric': 'l2',
        'verbosity': -1,
        'num_leaves': 128,            # 增加叶子数
        'learning_rate': 0.035,        # 降低学习率
        'n_estimators': 2000,         # 增加迭代次数
        'min_data_in_leaf': 20,       # 每个叶子最小样本数
        'feature_fraction': 0.8,      # 随机特征采样比例
        'bagging_fraction': 0.8,      # 随机样本采样比例
        'bagging_freq': 1,            # 每次迭代都进行bagging
        'seed': SEED
    }
    gbm = lgb.train(params, lgb_train)

    # fidelity
    y_hat = gbm.predict(X)
    fidelity_r2 = r2_score(y_svr, y_hat)
    print(f"数据集 {dataset_name} surrogate fidelity R² = {fidelity_r2:.4f}")

    # 保存记录
    fidelity_records.append({
        "dataset_name": dataset_name,
        "file_prefix": prefix,
        "fidelity_r2": fidelity_r2,
        "n_samples": df.shape[0]
    })

    # ========== SHAP 分析 ==========
    # 为每个数据集创建独立的输出文件夹
    dataset_folder = os.path.join(OUTPUT_FOLDER, dataset_name)
    os.makedirs(dataset_folder, exist_ok=True)
    
    print(f"📁 SHAP输出目录: {dataset_folder}")

    explainer = shap.TreeExplainer(gbm, feature_perturbation="tree_path_dependent")
    shap_values = explainer(X)

    # 保存SHAP相关文件
    np.save(os.path.join(dataset_folder, f"{dataset_name}_shap_values.npy"), shap_values.values)
    np.save(os.path.join(dataset_folder, f"{dataset_name}_base_values.npy"), shap_values.base_values)
    joblib.dump(shap_values, os.path.join(dataset_folder, f"{dataset_name}_shap_explanation.joblib"))
    
    # 保存代理模型
    gbm.save_model(os.path.join(dataset_folder, f"{dataset_name}_surrogate_model.txt"))

    # 特征重要性分析
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    
    # ========== 参考第二段代码：Top-10特征分析和绘图 ==========
    top10_idx = np.argsort(mean_abs_shap)[::-1][:10]
    top10_names = [ALL_FEATURES[i] for i in top10_idx]
    
    print(f"\n📊 Top 10 Features by SHAP Importance for {dataset_name}:")
    for i, (idx, name) in enumerate(zip(top10_idx, top10_names), 1):
        print(f"{i:2d}. {name}: {mean_abs_shap[idx]:.6f}")
    
    # 保存Top-10特征重要性排名
    df_top10 = pd.DataFrame({
        "Rank": range(1, 11),
        "Feature": top10_names,
        "MeanAbsSHAP": [mean_abs_shap[i] for i in top10_idx]
    })
    df_top10.to_csv(os.path.join(dataset_folder, f"{dataset_name}_top10_shap_features.csv"), index=False)
    
    # 保存完整特征重要性
    df_all_features = pd.DataFrame({
        "Feature": ALL_FEATURES,
        "MeanAbsSHAP": mean_abs_shap
    }).sort_values("MeanAbsSHAP", ascending=False)
    df_all_features.to_csv(os.path.join(dataset_folder, f"{dataset_name}_all_shap_features.csv"), index=False)

    # ========== 创建并保存Top-10 summary plot（参考第二段代码样式）==========
    # 提取Top-10的SHAP值和数据
    shap_values_top10 = shap_values.values[:, top10_idx]
    data_top10 = X[:, top10_idx]
    
    # 创建Top-10 summary plot（大字体设置）
    plt.figure(figsize=(12, 10))
    shap.summary_plot(
        shap_values_top10,
        data_top10,
        feature_names=top10_names,
        max_display=10,
        show=False,
        plot_size=None
    )
    plt.title(f'{dataset_name}', fontsize=TITLE_SIZE, fontweight='bold')
    plt.tight_layout()
    
    # 保存到SHAP输出目录
    shap_summary_png = os.path.join(dataset_folder, f"{dataset_name}_shap_summary_top10.png")
    shap_summary_pdf = os.path.join(dataset_folder, f"{dataset_name}_shap_summary_top10.pdf")
    plt.savefig(shap_summary_png, dpi=600, bbox_inches='tight')
    plt.savefig(shap_summary_pdf, bbox_inches='tight')
    plt.close()
    
    # 同时保存到统一的绘图目录
    drawing_summary_png = os.path.join(DRAWING_FOLDER, f"{dataset_name}_shap_summary_top10.png")
    drawing_summary_pdf = os.path.join(DRAWING_FOLDER, f"{dataset_name}_shap_summary_top10.pdf")
    plt.figure(figsize=(12, 10))
    shap.summary_plot(
        shap_values_top10,
        data_top10,
        feature_names=top10_names,
        max_display=10,
        show=False,
        plot_size=None
    )
    plt.title(f'{dataset_name}', fontsize=TITLE_SIZE, fontweight='bold')
    plt.tight_layout()
    plt.savefig(drawing_summary_png, dpi=600, bbox_inches='tight')
    plt.savefig(drawing_summary_pdf, bbox_inches='tight')
    plt.close()
    
    # 保存Top-10特征列表到绘图目录
    top10_csv_drawing = os.path.join(DRAWING_FOLDER, f"{dataset_name}_top10_features.csv")
    df_top10.to_csv(top10_csv_drawing, index=False)
    
    print(f"✅ Top-10 SHAP summary plot saved:")
    print(f"   SHAP dir: {shap_summary_png}")
    print(f"   Drawing dir: {drawing_summary_png}")

    # 生成并保存条形图（特征重要性）- 也使用Top-10
    plt.figure(figsize=(12, 10))
    df_top10_sorted = df_top10.sort_values("MeanAbsSHAP", ascending=True)
    plt.barh(range(len(df_top10_sorted)), df_top10_sorted["MeanAbsSHAP"])
    plt.yticks(range(len(df_top10_sorted)), df_top10_sorted["Feature"], fontsize=TICK_SIZE)
    plt.xlabel("Mean |SHAP value|", fontsize=FONT_SIZE)
    plt.title(f"Top 10 Feature Importance - {dataset_name}", fontsize=TITLE_SIZE, fontweight='bold')
    plt.tight_layout()
    
    # 保存条形图
    bar_png = os.path.join(dataset_folder, f"{dataset_name}_feature_importance_bar_top10.png")
    bar_pdf = os.path.join(dataset_folder, f"{dataset_name}_feature_importance_bar_top10.pdf")
    plt.savefig(bar_png, dpi=600, bbox_inches='tight')
    plt.savefig(bar_pdf, bbox_inches='tight')
    
    # 同时保存到绘图目录
    drawing_bar_png = os.path.join(DRAWING_FOLDER, f"{dataset_name}_feature_importance_bar_top10.png")
    drawing_bar_pdf = os.path.join(DRAWING_FOLDER, f"{dataset_name}_feature_importance_bar_top10.pdf")
    plt.savefig(drawing_bar_png, dpi=600, bbox_inches='tight')
    plt.savefig(drawing_bar_pdf, bbox_inches='tight')
    plt.close()

    # 生成并保存前2个重要特征的依赖图（可选，但保留）
    for idx in top10_idx[:2]:
        feat_name = ALL_FEATURES[idx]
        shap.dependence_plot(
            feat_name,
            shap_values.values,
            X,
            feature_names=ALL_FEATURES,
            show=False
        )
        plt.tight_layout()
        plt.savefig(os.path.join(dataset_folder, f"{dataset_name}_shap_dependence_{feat_name}.png"), dpi=300)
        plt.close()

    # 生成并保存力解释图（前3个样本）
    for sample_idx in range(min(3, X.shape[0])):
        force_html = os.path.join(dataset_folder, f"{dataset_name}_shap_force_sample{sample_idx}.html")
        force_plot_obj = shap.force_plot(
            shap_values.base_values[sample_idx],
            shap_values.values[sample_idx],
            X[sample_idx, :],
            feature_names=ALL_FEATURES
        )
        shap.save_html(force_html, force_plot_obj)

    print(f"✅ {dataset_name} 的 SHAP 分析完成，结果保存在：{dataset_folder}")

# ========== 汇总 fidelity R² ==========
if fidelity_records:
    fidelity_df = pd.DataFrame(fidelity_records)
    
    # 保存到SHAP输出目录
    summary_path = os.path.join(OUTPUT_FOLDER, "all_datasets_fidelity_summary.csv")
    fidelity_df.to_csv(summary_path, index=False)
    
    # 也保存到模型目录（保持向后兼容）
    model_summary_path = os.path.join(MODEL_FOLDER, "all_models_fidelity_summary.csv")
    fidelity_df.to_csv(model_summary_path, index=False)
    
    # 保存到绘图目录
    drawing_summary_path = os.path.join(DRAWING_FOLDER, "all_datasets_fidelity_summary.csv")
    fidelity_df.to_csv(drawing_summary_path, index=False)
    
    print(f"\n📊 所有数据集 fidelity R² 已统计完成")
    print(f"主要保存至: {summary_path}")
    print(f"备份保存至: {model_summary_path}")
    print(f"绘图目录保存至: {drawing_summary_path}")
    print("\n" + "="*50)
    print("Fidelity 统计结果:")
    print("="*50)
    print(fidelity_df.to_string(index=False))
    print("="*50)
    
    # 打印简要统计
    print(f"\n📈 Fidelity R² 统计:")
    print(f"  平均值: {fidelity_df['fidelity_r2'].mean():.4f}")
    print(f"  中位数: {fidelity_df['fidelity_r2'].median():.4f}")
    print(f"  标准差: {fidelity_df['fidelity_r2'].std():.4f}")
    print(f"  范围: {fidelity_df['fidelity_r2'].min():.4f} - {fidelity_df['fidelity_r2'].max():.4f}")
else:
    print("\n⚠️ 没有可统计的 fidelity R² 结果")

print(f"\n✨ 所有SHAP分析完成！输出目录结构:")
print(f"{OUTPUT_FOLDER}/")
for prefix in FILE_PREFIXES:
    dataset_name = prefix.replace("-filtered", "")
    print(f"  ├── {dataset_name}/")
    print(f"  │   ├── {dataset_name}_shap_values.npy")
    print(f"  │   ├── {dataset_name}_base_values.npy")
    print(f"  │   ├── {dataset_name}_shap_explanation.joblib")
    print(f"  │   ├── {dataset_name}_surrogate_model.txt")
    print(f"  │   ├── {dataset_name}_top10_shap_features.csv")
    print(f"  │   ├── {dataset_name}_all_shap_features.csv")
    print(f"  │   ├── {dataset_name}_shap_summary_top10.png")
    print(f"  │   ├── {dataset_name}_shap_summary_top10.pdf")
    print(f"  │   ├── {dataset_name}_feature_importance_bar_top10.png")
    print(f"  │   ├── {dataset_name}_feature_importance_bar_top10.pdf")
    print(f"  │   ├── {dataset_name}_shap_dependence_*.png")
    print(f"  │   └── {dataset_name}_shap_force_sample*.html")
print(f"  └── all_datasets_fidelity_summary.csv")
print(f"\n{DRAWING_FOLDER}/")
for prefix in FILE_PREFIXES:
    dataset_name = prefix.replace("-filtered", "")
    print(f"  ├── {dataset_name}_shap_summary_top10.png")
    print(f"  ├── {dataset_name}_shap_summary_top10.pdf")
    print(f"  ├── {dataset_name}_feature_importance_bar_top10.png")
    print(f"  ├── {dataset_name}_feature_importance_bar_top10.pdf")
    print(f"  ├── {dataset_name}_top10_features.csv")
print(f"  └── all_datasets_fidelity_summary.csv")

print(f"\n🎉 Top-10 SHAP summary plot generation completed for all datasets!")
print(f"📊 Total features per dataset: {len(ALL_FEATURES)}")
print(f"📁 Drawing output directory: {DRAWING_FOLDER}")


===== 开始 SHAP 分析: AM-III (AM-III-filtered) =====
数据集 AM-III surrogate fidelity R² = 0.9998
📁 SHAP输出目录: ./3-shap-other4/AM-III

📊 Top 10 Features by SHAP Importance for AM-III:
 1. logP: 5.199603
 2. fp_842: 2.198962
 3. MolWt: 1.304684
 4. fp_707: 1.077610
 5. fp_73: 1.001477
 6. TPSA: 0.810213
 7. fp_902: 0.654150
 8. fp_672: 0.635069
 9. fp_896: 0.609616
10. fp_200: 0.579743
✅ Top-10 SHAP summary plot saved:
   SHAP dir: ./3-shap-other4/AM-III/AM-III_shap_summary_top10.png
   Drawing dir: ./3-shap-other4/Top10/AM-III_shap_summary_top10.png
✅ AM-III 的 SHAP 分析完成，结果保存在：./3-shap-other4/AM-III

===== 开始 SHAP 分析: AM-IV (AM-IV-filtered) =====
数据集 AM-IV surrogate fidelity R² = 0.9986
📁 SHAP输出目录: ./3-shap-other4/AM-IV

📊 Top 10 Features by SHAP Importance for AM-IV:
 1. logP: 3.104805
 2. col316: 1.367522
 3. col350: 1.160857
 4. MolWt: 1.117121
 5. col403: 0.866064
 6. fp_842: 0.702437
 7. col408: 0.684078
 8. fp_818: 0.611506
 9. fp_378: 0.584746
10. fp_121: 0.583529
✅ Top-10 SHAP summary 